In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/tanushkumaryadav/full-heusler/check_fh.csv


In [2]:
!pip install pymatgen

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 656.5 kB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.1/829.1 kB 1.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 3.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 7.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 332.3/332.3 kB 6.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 13.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 962.5/962.5 kB 15.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 2.9 MB/s eta 0:00:00
  Created wheel for bibtexparser: filename=bibtexparser-1.4.4-py3-none-any.whl size=43609 sha256=e7093b19ee21e

In [26]:
df=pd.read_csv("/kaggle/input/datasets/tanushkumaryadav/full-heusler/check_fh.csv")

In [27]:
df.shape

(1725, 2)

In [28]:
df.head(20)

,formula,choice
0,V2ScAl,1
1,V2ScGa,1
2,V2ScIn,1
3,V2ScSi,1
4,V2ScGe,1
5,V2ScSn,1
6,V2ScP,1
7,V2ScAs,1
8,V2ScSb,1
9,V2TiAl,1


In [29]:
from pymatgen.core import Composition

def has_three_unique_elements(formula):
    return len(Composition(formula).elements) == 3

df = df[df["formula"].apply(has_three_unique_elements)]

In [30]:
df.shape

(1671, 2)

In [34]:
from pymatgen.core import Composition, Element
def parse_mol(formula):
    com=Composition(formula)
    elements = list(com.get_el_amt_dict().keys())
    amounts = list(com.get_el_amt_dict().values())
    B=''
    for el, amt in com.get_el_amt_dict().items():
        if amt == 2:
            B=el
    elements.remove(B)
    X=[Element(elements[0]).X,Element(elements[1]).X]
    c,a = sorted(X)
    A = elements[X.index(a)]
    C=elements[X.index(c)]
    return A,B,C
parse_mol("Mn1Co2Ga1")

('Ga', 'Co', 'Mn')

In [35]:
descriptors = {'t_s': [], 't_p': [], 't_d': [], 't_all': [],'r_AB': [], 'r_AC': [],'r_BC': [], 'en_A': [], 'en_B': [], 'en_C': [],'g_A': [], 
               'g_B': [], 'g_C': [], 'r_A': [], 'r_B': [], 'r_C': [],'m_A': [], 'm_B': [], 'm_C': [],
                'p_A': [], 'p_B': [], 'p_C': []}
def Descriptors(A,B,C):
    el_A=Element(A)
    el_B=Element(B)
    el_C=Element(C)
    q_s,q_p,q_d=0,0,0
    for atom,mul in [(A,1),(B,2),(C,1)]:
        el=Element(atom)
        structure=el.full_electronic_structure
        n_max = max(n for n, l, q in structure)
        for n,l,q in el.full_electronic_structure:
            if(l=='s' and n==n_max):
                q_s+=mul*q
            if(l=='p' and n==n_max):
                q_p+=mul*q
            if(l=='d' and n==n_max-1 and el.group <= 12):
                q_d+=mul*q
    descriptors['t_s'].append(q_s)
    descriptors['t_p'].append(q_p)
    descriptors['t_d'].append(q_d)
    descriptors['t_all'].append(q_s+q_p+q_d)
    R_AB=abs(el_A.atomic_radius-el_B.atomic_radius)
    descriptors['r_AB'].append(R_AB)
    descriptors['en_B'].append(el_B.X)
    descriptors['en_C'].append(el_C.X)
    descriptors['en_A'].append(el_A.X)
    descriptors['g_A'].append(el_A.group)
    descriptors['g_B'].append(el_B.group)
    descriptors['g_C'].append(el_C.group)
    R_AC=abs(el_A.atomic_radius-el_C.atomic_radius)
    descriptors['r_AC'].append(R_AC)
    R_BC=abs(el_B.atomic_radius-el_C.atomic_radius)
    descriptors['r_BC'].append(R_BC)
    descriptors['r_C'].append(el_C.atomic_radius)
    descriptors['r_B'].append(el_B.atomic_radius)
    descriptors['r_A'].append(el_A.atomic_radius)
    descriptors['m_C'].append(el_C.atomic_mass)
    descriptors['m_B'].append(el_B.atomic_mass)
    descriptors['m_A'].append(el_A.atomic_mass)
    descriptors['p_A'].append(el_A.row)
    descriptors['p_B'].append(el_B.row)
    descriptors['p_C'].append(el_C.row)
    


In [36]:
for formula in df['formula']:
    A,B,C=parse_mol(formula)
    Descriptors(A,B,C)

In [37]:
descriptor_df = pd.DataFrame(descriptors)

In [38]:
descriptor_df.shape

(1671, 22)

In [39]:
descriptor_df.head()

,t_s,t_p,t_d,t_all,r_AB,r_AC,r_BC,en_A,en_B,en_C,...,g_C,r_A,r_B,r_C,m_A,m_B,m_C,p_A,p_B,p_C
0,8,1,7,16,0.10,0.35,0.25,1.61,1.63,1.36,...,3,1.25,1.35,1.6,26.981539,50.9415,44.955912,3,4,4
1,8,1,7,16,0.05,0.30,0.25,1.81,1.63,1.36,...,3,1.30,1.35,1.6,69.723000,50.9415,44.955912,4,4,4
2,8,1,7,16,0.20,0.05,0.25,1.78,1.63,1.36,...,3,1.55,1.35,1.6,114.818000,50.9415,44.955912,5,4,4
3,8,2,7,17,0.25,0.50,0.25,1.90,1.63,1.36,...,3,1.10,1.35,1.6,28.085500,50.9415,44.955912,3,4,4
4,8,2,7,17,0.10,0.35,0.25,2.01,1.63,1.36,...,3,1.25,1.35,1.6,72.640000,50.9415,44.955912,4,4,4


In [40]:
df_final = pd.concat([df.reset_index(drop=True),descriptor_df.reset_index(drop=True)],axis=1)

In [41]:
X = df_final.drop(columns=['formula', 'choice'])
y = df_final['choice']

In [42]:
print(X.shape)
print(y.shape)

(1671, 22)
(1671,)


In [43]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,stratify=y,random_state=42)

In [44]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier

param_dist = {
    'n_estimators': [200, 500, 1000],
    'max_depth': [5, 10, 15, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2'],
    'class_weight': ['balanced']
}

rf = RandomForestClassifier(random_state=42)

search = RandomizedSearchCV(
    rf,
    param_distributions=param_dist,
    n_iter=50,
    scoring='recall',
    cv=5,
    n_jobs=-1,
    random_state=42
)

search.fit(X_train, y_train)

print(search.best_params_)

{'n_estimators': 200, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': 5, 'class_weight': 'balanced'}


In [46]:
best_params = search.best_params_
print(best_params)

{'n_estimators': 200, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': 5, 'class_weight': 'balanced'}


In [47]:
from sklearn.metrics import (classification_report,confusion_matrix,roc_auc_score)

rf_best = RandomForestClassifier(**best_params,random_state=42,n_jobs=-1)

rf_best.fit(X_train, y_train)

prob = rf_best.predict_proba(X_test)[:,1]

for t in [0.2,0.3,0.4,0.5]:
    y_pred = (prob > t).astype(int)

    print(classification_report(y_test, y_pred))
    
    print(confusion_matrix(y_test, y_pred))
    
    print(roc_auc_score(y_test,rf_best.predict_proba(X_test)[:, 1]))

              precision    recall  f1-score   support

           0       1.00      0.85      0.92       230
           1       0.75      1.00      0.86       105

    accuracy                           0.90       335
   macro avg       0.88      0.92      0.89       335
weighted avg       0.92      0.90      0.90       335

[[195  35]
 [  0 105]]
0.9884057971014493
              precision    recall  f1-score   support

           0       1.00      0.92      0.96       230
           1       0.85      1.00      0.92       105

    accuracy                           0.94       335
   macro avg       0.92      0.96      0.94       335
weighted avg       0.95      0.94      0.94       335

[[211  19]
 [  0 105]]
0.9884057971014493
              precision    recall  f1-score   support

           0       1.00      0.95      0.98       230
           1       0.91      1.00      0.95       105

    accuracy                           0.97       335
   macro avg       0.95      0.98      0.96 

In [48]:
rf_best.fit(X_train,y_train)

y_pred = rf_best.predict(X_test)

print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       1.00      0.97      0.99       230
           1       0.95      1.00      0.97       105

    accuracy                           0.98       335
   macro avg       0.97      0.99      0.98       335
weighted avg       0.98      0.98      0.98       335



In [51]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(rf_best,X,y,cv=5,scoring='f1')

print(scores)
print(scores.mean())

[0.87       0.95890411 0.98113208 0.96744186 0.54901961]
0.8652995306737985


In [54]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

skf = StratifiedKFold(n_splits=20,shuffle=True,random_state=42)

scores = cross_val_score(rf_best,X,y,cv=skf,scoring='f1')

print(scores)
print(scores.mean())
print(scores.std())

[0.94736842 0.93103448 0.92857143 0.94545455 0.94545455 0.96296296
 1.         0.98113208 0.98113208 0.94545455 0.98113208 0.98113208
 1.         0.98113208 0.94545455 1.         0.94545455 1.
 0.96296296 0.98113208]
0.9673482719205762
0.02351719402329813


In [55]:
import pandas as pd

importance = pd.DataFrame({'feature': X.columns,'importance': rf_best.feature_importances_})

importance = importance.sort_values('importance',ascending=False)

print(importance.head(22))

   feature  importance
1      t_p    0.177722
14     r_B    0.103108
11     g_B    0.090641
18     m_C    0.082489
2      t_d    0.078289
17     m_B    0.074284
7     en_A    0.067852
21     p_C    0.065982
6     r_BC    0.059979
9     en_C    0.042858
12     g_C    0.042776
16     m_A    0.027888
8     en_B    0.018426
19     p_A    0.016938
10     g_A    0.016450
20     p_B    0.010338
15     r_C    0.009090
4     r_AB    0.009084
3    t_all    0.002748
5     r_AC    0.002381
13     r_A    0.000489
0      t_s    0.000187
